# 01 — Data
Load factors and returns, inspect the dataset, check for missing values and basic statistics.

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

plt.rcParams.update({'figure.dpi': 130, 'axes.spines.top': False,
                     'axes.spines.right': False, 'font.size': 11})


## Load data
Set `use_synthetic=False` once `data/raw/` is populated by `fetch_data.py`.

In [ ]:
from src.data import load_data

factors, returns = load_data(use_synthetic=False)   # change to True for synthetic

print(f"Factors : {factors.shape}  {factors.index[0].date()} → {factors.index[-1].date()}")
print(f"Returns : {returns.shape}")
print()
print("Factor columns:", list(factors.columns))
print("Asset  columns:", list(returns.columns))


## Factor summary statistics

In [ ]:
factors.describe().round(4)


## Returns summary statistics

In [ ]:
returns.describe().round(4)


## Missing values

In [ ]:
missing = returns.isna().sum()
print("Tickers with missing months:")
print(missing[missing > 0].sort_values(ascending=False))
print(f"\nTotal missing: {missing.sum()} / {returns.size:,} ({missing.sum()/returns.size*100:.2f}%)")


## Factor time series

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(12, 8), sharex=True)
factor_cols = [c for c in factors.columns if c != 'RF']

for ax, col in zip(axes.flatten(), factor_cols):
    ax.plot(factors.index, factors[col], linewidth=0.8, color='steelblue')
    ax.axhline(0, linewidth=0.5, color='black', linestyle='--', alpha=0.4)
    ax.set_title(col, fontsize=11)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

fig.suptitle('Fama–French Factor Returns (Monthly)', fontsize=13)
plt.tight_layout()
plt.savefig('../results/figures/factor_series.png', bbox_inches='tight')
plt.show()


## Return correlation heatmap

In [ ]:
import numpy as np

corr = returns.corr()
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)))
ax.set_yticks(range(len(corr)))
ax.set_xticklabels(corr.columns, rotation=90, fontsize=8)
ax.set_yticklabels(corr.columns, fontsize=8)
plt.colorbar(im, ax=ax, label='Pearson correlation')
ax.set_title('Cross-asset return correlation matrix', fontsize=12)
plt.tight_layout()
plt.savefig('../results/figures/return_corr.png', bbox_inches='tight')
plt.show()
print(f"Mean pairwise correlation: {corr.values[np.triu_indices(len(corr), k=1)].mean():.3f}")


## Distribution of monthly returns

In [ ]:
all_rets = returns.stack().dropna()
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].hist(all_rets, bins=80, color='steelblue', edgecolor='none', alpha=0.8)
axes[0].set_title('Distribution of monthly returns (all assets)')
axes[0].set_xlabel('Monthly return')
axes[0].set_ylabel('Count')

axes[1].plot(returns.mean(axis=1).rolling(12).mean() * 12, label='Mean (ann.)', color='steelblue')
axes[1].fill_between(returns.index,
                     (returns.mean(axis=1) - returns.std(axis=1)).rolling(12).mean() * 12,
                     (returns.mean(axis=1) + returns.std(axis=1)).rolling(12).mean() * 12,
                     alpha=0.2, color='steelblue')
axes[1].axhline(0, linewidth=0.6, linestyle='--', color='black', alpha=0.4)
axes[1].set_title('Cross-sectional mean return (12-month rolling, ann.)')
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
axes[1].legend()

plt.tight_layout()
plt.savefig('../results/figures/return_dist.png', bbox_inches='tight')
plt.show()

print(f"Mean monthly return : {all_rets.mean():.4f}  ({all_rets.mean()*12:.3f} ann.)")
print(f"Std monthly return  : {all_rets.std():.4f}  ({all_rets.std()*12**0.5:.3f} ann.)")
print(f"Skewness            : {all_rets.skew():.3f}")
print(f"Kurtosis            : {all_rets.kurt():.3f}")
